# Cloud Provider Analytics · Notebook 01 — Exploración y fundación del Data Lake (Landing → Bronze)

**Big Data · ITBA · 2C 2026 · Primera entrega (diseño y fundación)**

Este notebook es la *evidencia mínima de lectura y exploración de datos* de la primera entrega. No busca implementar el pipeline completo (eso es la segunda entrega), sino **validar con Spark las decisiones del documento de diseño**:

| Sección | Qué demuestra | Vínculo con el diseño |
|---|---|---|
| 1. Setup | SparkSession y configuración externalizada | Reproducibilidad |
| 2. Landing | Inventario de archivos; Landing es **inmutable** (solo lectura) | Zonas del Data Lake |
| 3. Esquemas explícitos | Nada de `inferSchema`: tipos declarados y campos opcionales de v2 | Bronze = tipificación explícita |
| 4. Perfil de calidad | Nulos, tipos ambiguos, evolución de esquema, negativos, outliers, monedas, late data | Veracidad / reglas de calidad |
| 5. Bronze | Parquet particionado + `ingest_ts` + `source_file` + dedupe; re-ejecución idempotente | Diseño de Bronze |
| 6. MapReduce ↔ Spark | La transformación batch de referencia en RDD (map/reduce) y en DataFrame, con el plan físico | Flujo batch de referencia |

**Cómo correrlo en Google Colab:** subir `cloud_provider_challenge_dataset_v1.zip` al panel *Archivos* (ícono de carpeta) y ejecutar *Entorno de ejecución → Ejecutar todo*.

## 1. Setup del entorno

In [1]:
# Colab suele traer PySpark instalado; si no, se instala (versión fijada para reproducibilidad).
try:
    import pyspark
except ImportError:
    !pip install -q pyspark==3.5.3
    import pyspark
print("PySpark", pyspark.__version__)

PySpark 4.0.4


In [2]:
import os, zipfile, json, glob, datetime
from pyspark.sql import SparkSession, functions as F, types as T

# ---- Configuración externalizada (en el repo vive en config/pipeline.yaml) ----
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
CONFIG = {
    "zip_name":      "cloud_provider_challenge_dataset_v1.zip",
    "lake_root":     f"{BASE}/datalake",
    "landing":       f"{BASE}/datalake/landing",
    "bronze":        f"{BASE}/datalake/bronze",
    "evidence":      f"{BASE}/evidence",
    "shuffle_partitions": 8,   # dataset chico: 200 (default) crearía cientos de tareas vacías
}
os.makedirs(CONFIG["evidence"], exist_ok=True)

spark = (SparkSession.builder
         .appName("cloud-provider-analytics-01-exploracion")
         .config("spark.sql.shuffle.partitions", CONFIG["shuffle_partitions"])
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic")  # re-ejecución idempotente por partición
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| master:", spark.sparkContext.master,
      "| paralelismo por defecto:", spark.sparkContext.defaultParallelism)

Spark 4.0.4 | master: local[*] | paralelismo por defecto: 2


## 2. Landing: descompresión e inventario

Landing es la zona **raw inmutable**: se descomprime una vez y a partir de ahí **solo se lee**. Ninguna celda de este notebook escribe en `landing/`.

In [3]:
zip_path = os.path.join(BASE, CONFIG["zip_name"])
if not os.path.exists(os.path.join(CONFIG["landing"], "customers_orgs.csv")):
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Subí {CONFIG['zip_name']} al panel de archivos de Colab ({BASE}).")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BASE)          # el zip ya trae la estructura datalake/landing/...
print("Landing listo en", CONFIG["landing"])

# Inventario físico: cantidad de archivos y tamaño por fuente
inventario = []
for p in sorted(glob.glob(CONFIG["landing"] + "/*")):
    files = glob.glob(p + "/*") if os.path.isdir(p) else [p]
    size = sum(os.path.getsize(f) for f in files)
    inventario.append((os.path.basename(p), len(files), round(size / 1024, 1)))
print(f"{'fuente':<28}{'archivos':>9}{'KB':>10}")
for n, c, kb in inventario:
    print(f"{n:<28}{c:>9}{kb:>10}")
total_kb = sum(i[2] for i in inventario)
print(f"{'TOTAL':<28}{sum(i[1] for i in inventario):>9}{total_kb:>10}")
ev_files = [i for i in inventario if i[0] == "usage_events_stream"][0]
print(f"\nTamaño promedio de un archivo de eventos: {ev_files[2]/ev_files[1]:.0f} KB "
      "→ muy por debajo de un bloque HDFS (128 MB): problema de 'small files' a considerar en Bronze.")

Landing listo en /content/datalake/landing
fuente                       archivos        KB
billing_monthly.csv                 1      15.5
customers_orgs.csv                  1       7.6
marketing_touches.csv               1      99.6
nps_surveys.csv                     1       3.9
resources.csv                       1      35.8
support_tickets.csv                 1      69.3
usage_events_stream               120   12624.5
users.csv                           1      69.0
TOTAL                             127   12925.2

Tamaño promedio de un archivo de eventos: 105 KB → muy por debajo de un bloque HDFS (128 MB): problema de 'small files' a considerar en Bronze.


## 3. Esquemas explícitos

Decisiones:
- **No se usa `inferSchema`**: en streaming no está permitido y en batch oculta los problemas de tipos.
- En eventos, `value` se lee como **string** (`value_raw`) porque llega a veces como número y a veces como texto; el casteo se hace después con *fallback controlado* y se conserva el valor original para trazabilidad.
- `carbon_kg` y `genai_tokens` se declaran **opcionales**: solo existen en `schema_version = 2`.
- En los CSV, los booleanos y fechas se leen como string en Bronze y se tipan con reglas explícitas (evita que un valor mal formado haga fallar toda la carga).

In [4]:
S = T.StringType(); D = T.DoubleType(); I = T.IntegerType(); L = T.LongType()
def schema(*cols): return T.StructType([T.StructField(n, t, True) for n, t in cols])

SCHEMAS = {
    "customers_orgs":    schema(("org_id",S),("org_name",S),("industry",S),("hq_region",S),("plan_tier",S),
                                ("is_enterprise",S),("signup_date",S),("sales_rep",S),("lifecycle_stage",S),
                                ("marketing_source",S),("nps_score",D)),
    "users":             schema(("user_id",S),("org_id",S),("email",S),("role",S),("active",S),
                                ("created_at",S),("last_login",S)),
    "resources":         schema(("resource_id",S),("org_id",S),("service",S),("region",S),("created_at",S),
                                ("state",S),("tags_json",S)),
    "support_tickets":   schema(("ticket_id",S),("org_id",S),("category",S),("severity",S),("created_at",S),
                                ("resolved_at",S),("csat",D),("sla_breached",S)),
    "marketing_touches": schema(("touch_id",S),("org_id",S),("campaign",S),("channel",S),("timestamp",S),
                                ("clicked",S),("converted",S)),
    "nps_surveys":       schema(("org_id",S),("survey_date",S),("nps_score",D),("comment",S)),
    "billing_monthly":   schema(("invoice_id",S),("org_id",S),("month",S),("subtotal",D),("credits",D),
                                ("taxes",D),("currency",S),("exchange_rate_to_usd",D)),
}
EVENTS_SCHEMA = schema(("event_id",S),("timestamp",S),("org_id",S),("resource_id",S),("service",S),
                       ("region",S),("metric",S),("value",S),("unit",S),("cost_usd_increment",D),
                       ("schema_version",I),("carbon_kg",D),("genai_tokens",L))

def read_csv(name):
    return (spark.read.option("header", True).option("mode", "PERMISSIVE")
            .schema(SCHEMAS[name]).csv(f"{CONFIG['landing']}/{name}.csv"))

raw = {name: read_csv(name) for name in SCHEMAS}
raw_events = (spark.read.schema(EVENTS_SCHEMA)
              .json(f"{CONFIG['landing']}/usage_events_stream/*.jsonl")
              .withColumnRenamed("value", "value_raw")
              .withColumn("source_file", F.input_file_name()))

# Spark es 'lazy': definir la lectura no lee datos. El único job que puede aparecer acá es el LISTADO
# de los 120 archivos (Spark lista en paralelo cuando hay más de 32 rutas). Recién .count() lee los datos.
print("Plan de la lectura de eventos (definido, todavía sin leer datos):")
raw_events.explain(mode="simple")

Plan de la lectura de eventos (definido, todavía sin leer datos):
== Physical Plan ==
*(1) Project [event_id#52, timestamp#53, org_id#54, resource_id#55, service#56, region#57, metric#58, value#59 AS value_raw#66, unit#60, cost_usd_increment#61, schema_version#62, carbon_kg#63, genai_tokens#64L, input_file_name() AS source_file#67]
+- FileScan json [event_id#52,timestamp#53,org_id#54,resource_id#55,service#56,region#57,metric#58,value#59,unit#60,cost_usd_increment#61,schema_version#62,carbon_kg#63,genai_tokens#64L] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(120 paths)[file:/content/datalake/landing/usage_events_stream/events_part_0000...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<event_id:string,timestamp:string,org_id:string,resource_id:string,service:string,region:st...




In [5]:
conteos = {name: df.count() for name, df in raw.items()}
conteos["usage_events_stream"] = raw_events.count()
for k, v in conteos.items():
    print(f"{k:<22}{v:>8,} filas")
print("\nParticiones de lectura de eventos:", raw_events.rdd.getNumPartitions(),
      "(Spark agrupa los 120 archivos chicos en pocas particiones de entrada)")

customers_orgs              80 filas
users                      800 filas
resources                  400 filas
support_tickets          1,000 filas
marketing_touches        1,500 filas
nps_surveys                 92 filas
billing_monthly            240 filas
usage_events_stream     43,200 filas

Particiones de lectura de eventos: 4 (Spark agrupa los 120 archivos chicos en pocas particiones de entrada)


## 4. Perfil de calidad de datos

Objetivo: medir (no suponer) los problemas que la consigna anticipa, para convertirlos en **reglas de calidad** de Bronze/Silver.

In [6]:
def nulos(df):
    exprs = [F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
             for c in df.columns if c != "source_file"]
    row = df.agg(*exprs).collect()[0].asDict()
    return {k: v for k, v in row.items() if v}

perfil = {"conteos": conteos, "nulos": {}}
for name, df in list(raw.items()) + [("usage_events_stream", raw_events)]:
    perfil["nulos"][name] = nulos(df)
    print(f"{name:<22} nulos → {perfil['nulos'][name] or '—'}")

customers_orgs         nulos → {'nps_score': 11}
users                  nulos → {'last_login': 139}
resources              nulos → {'tags_json': 83}
support_tickets        nulos → {'resolved_at': 240, 'csat': 254}
marketing_touches      nulos → —
nps_surveys            nulos → {'nps_score': 19, 'comment': 10}
billing_monthly        nulos → {'credits': 137}
usage_events_stream    nulos → {'value_raw': 877, 'unit': 2075, 'carbon_kg': 10800, 'genai_tokens': 40068}


### 4.1 Eventos: evolución de esquema (v1 → v2)

In [7]:
ev = raw_events.withColumn("event_ts", F.to_timestamp("timestamp")).withColumn("event_date", F.to_date("event_ts"))

por_version = (ev.groupBy("schema_version")
               .agg(F.count("*").alias("eventos"), F.min("event_date").alias("desde"), F.max("event_date").alias("hasta"),
                    F.sum(F.col("carbon_kg").isNotNull().cast("int")).alias("con_carbon_kg"),
                    F.sum(F.col("genai_tokens").isNotNull().cast("int")).alias("con_genai_tokens"))
               .orderBy("schema_version"))
por_version.show()

(ev.filter(F.col("schema_version") == 2).groupBy("service")
   .agg(F.count("*").alias("eventos_v2"), F.sum(F.col("genai_tokens").isNotNull().cast("int")).alias("con_tokens"))
   .orderBy("service").show())

+--------------+-------+----------+----------+-------------+----------------+
|schema_version|eventos|     desde|     hasta|con_carbon_kg|con_genai_tokens|
+--------------+-------+----------+----------+-------------+----------------+
|             1|  10800|2025-07-03|2025-07-17|            0|               0|
|             2|  32400|2025-07-18|2025-08-31|        32400|            3132|
+--------------+-------+----------+----------+-------------+----------------+

+----------+----------+----------+
|   service|eventos_v2|con_tokens|
+----------+----------+----------+
| analytics|      3450|         0|
|   compute|      9369|         0|
|  database|      5559|         0|
|     genai|      3132|      3132|
|networking|      5217|         0|
|   storage|      5673|         0|
+----------+----------+----------+



### 4.2 Eventos: tipos ambiguos en `value` y consistencia `value`/`unit`

In [8]:
ev_typed = ev.withColumn("value_num", F.col("value_raw").cast("double"))   # fallback controlado: si no castea → null

# Detectamos cuáles llegaron como texto leyendo el JSON crudo con 'value' como string y comparando contra el tipo original
tipos_value = (spark.read.text(f"{CONFIG['landing']}/usage_events_stream/*.jsonl")
    .select(F.when(F.col("value").rlike('"value": "'), "texto (\"120.0\")")
             .when(F.col("value").rlike('"value": null'), "null")
             .otherwise("numérico").alias("tipo_de_value"))
    .groupBy("tipo_de_value").count())
tipos_value.show(truncate=False)

print("¿Todos los textos castean a número? →",
      ev_typed.filter(F.col("value_raw").isNotNull() & F.col("value_num").isNull()).count() == 0)

(ev_typed.groupBy("metric", "unit").count().orderBy("metric", "unit").show())
sin_unit = ev_typed.filter(F.col("value_num").isNotNull() & F.col("unit").isNull()).count()
print(f"Eventos con value pero SIN unit (regla de calidad de la consigna): {sin_unit:,}")

+---------------+-----+
|tipo_de_value  |count|
+---------------+-----+
|null           |877  |
|numérico       |41014|
|texto ("120.0")|1309 |
+---------------+-----+

¿Todos los textos castean a número? → True
+----------------+--------+-----+
|          metric|    unit|count|
+----------------+--------+-----+
|       cpu_hours|    NULL|  505|
|       cpu_hours|   hours|10187|
|        requests|    NULL|  916|
|        requests|   count|18596|
|storage_gb_hours|    NULL|  654|
|storage_gb_hours|gb_hours|12342|
+----------------+--------+-----+

Eventos con value pero SIN unit (regla de calidad de la consigna): 2,038


### 4.3 Eventos: costos negativos y outliers (base para el umbral de anomalías)

In [9]:
c = "cost_usd_increment"
q = ev.approxQuantile(c, [0.01, 0.25, 0.5, 0.75, 0.99, 0.999], 0.0001)
stats = ev.agg(F.mean(c).alias("media"), F.stddev(c).alias("desvio"), F.min(c).alias("min"), F.max(c).alias("max")).collect()[0]
print("Percentiles p1, p25, p50, p75, p99, p99.9:", [round(x, 3) for x in q])
print({k: round(v, 3) for k, v in stats.asDict().items()})

negativos = ev.filter(F.col(c) < 0).count()
bajo_tol  = ev.filter(F.col(c) < -0.01).count()
print(f"Costos negativos: {negativos:,}  |  por debajo de la tolerancia -0.01 (regla de la consigna): {bajo_tol:,}")

# Vista previa de dos métodos de anomalía que se evaluarán en Silver/Gold
mediana = q[2]
mad = ev.select(F.abs(F.col(c) - mediana).alias("d")).approxQuantile("d", [0.5], 0.0001)[0]
print(f"z-score > 3  → {ev.filter(F.abs((F.col(c) - stats['media']) / stats['desvio']) > 3).count():,} eventos")
print(f"MAD robusto > 3.5 → {ev.filter(0.6745 * F.abs(F.col(c) - mediana) / mad > 3.5).count():,} eventos (MAD = {mad:.3f})")
print("→ Con un umbral GLOBAL, z-score marca muy pocos y MAD marca demasiados: el costo depende del servicio/métrica.")

# Por eso el umbral se calcula POR GRUPO (servicio, métrica): así se compara cada evento con su propia distribución
w = ev.groupBy("service", "metric").agg(
        F.percentile_approx(c, 0.5).alias("med"),
        F.percentile_approx(c, 0.99).alias("p99"))
por_grupo = (ev.join(w, ["service", "metric"])
               .withColumn("dev", F.abs(F.col(c) - F.col("med"))))
mad_g = por_grupo.groupBy("service", "metric").agg(F.percentile_approx("dev", 0.5).alias("mad"))
flag = (por_grupo.join(mad_g, ["service", "metric"])
          .withColumn("mad_score", 0.6745 * F.col("dev") / F.col("mad"))
          .groupBy("service", "metric")
          .agg(F.count("*").alias("eventos"),
               F.round(F.first("med"), 3).alias("mediana"),
               F.round(F.first("mad"), 3).alias("mad"),
               F.sum((F.col("mad_score") > 3.5).cast("int")).alias("anom_MAD>3.5"),
               F.sum((F.col(c) > F.col("p99")).cast("int")).alias("sobre_p99"))
          .orderBy("service", "metric"))
flag.show(20)

Percentiles p1, p25, p50, p75, p99, p99.9: [0.0, 0.129, 1.004, 6.115, 16.693, 103.608]
{'media': 3.413, 'desvio': 7.924, 'min': -154.461, 'max': 317.431}
Costos negativos: 216  |  por debajo de la tolerancia -0.01 (regla de la consigna): 211
z-score > 3  → 70 eventos
MAD robusto > 3.5 → 10,958 eventos (MAD = 0.975)
→ Con un umbral GLOBAL, z-score marca muy pocos y MAD marca demasiados: el costo depende del servicio/métrica.
+----------+----------------+-------+-------+-----+------------+---------+
|   service|          metric|eventos|mediana|  mad|anom_MAD>3.5|sobre_p99|
+----------+----------------+-------+-------+-----+------------+---------+
| analytics|       cpu_hours|   1148|  0.128|0.071|           6|       11|
| analytics|        requests|   2075|   8.35| 0.96|          21|       20|
| analytics|storage_gb_hours|   1367|  0.702|0.384|           6|       13|
|   compute|       cpu_hours|   3143|   0.14|0.084|          13|       31|
|   compute|        requests|   5570|   9.52|1.

### 4.4 Eventos: duplicados y *late data*

Si los archivos llegaran ordenados en el tiempo, cada archivo cubriría un tramo corto. Medimos qué rango de fechas trae cada archivo: esto define cómo configurar el **watermark** del streaming en la segunda entrega.

In [10]:
dups = ev.groupBy("event_id").count().filter("count > 1").count()
print(f"event_id nulos: {ev.filter(F.col('event_id').isNull()).count()}  |  event_id duplicados: {dups}")

por_archivo = (ev.withColumn("archivo", F.regexp_extract("source_file", r"(events_part_\d+)", 1))
                 .groupBy("archivo")
                 .agg(F.min("event_ts").alias("min_ts"), F.max("event_ts").alias("max_ts"), F.count("*").alias("eventos"))
                 .withColumn("rango_dias", F.round((F.unix_timestamp("max_ts") - F.unix_timestamp("min_ts")) / 86400, 1))
                 .orderBy("archivo"))
por_archivo.show(5, truncate=False)
por_archivo.agg(F.min("rango_dias").alias("rango_min_dias"), F.avg("rango_dias").alias("rango_prom_dias")).show()
print("→ Cada micro-lote trae eventos de TODO el período: el orden de llegada no sigue al event time.")

event_id nulos: 0  |  event_id duplicados: 0
+----------------+-------------------+-------------------+-------+----------+
|archivo         |min_ts             |max_ts             |eventos|rango_dias|
+----------------+-------------------+-------------------+-------+----------+
|events_part_0000|2025-07-03 00:08:00|2025-08-31 16:32:00|360    |59.7      |
|events_part_0001|2025-07-03 02:38:00|2025-08-31 23:51:00|360    |59.9      |
|events_part_0002|2025-07-03 14:16:00|2025-08-31 20:47:00|360    |59.3      |
|events_part_0003|2025-07-03 00:48:00|2025-08-31 23:51:00|360    |60.0      |
|events_part_0004|2025-07-03 10:43:00|2025-08-31 20:40:00|360    |59.4      |
+----------------+-------------------+-------------------+-------+----------+
only showing top 5 rows
+--------------+---------------+
|rango_min_dias|rango_prom_dias|
+--------------+---------------+
|          59.0|        59.6875|
+--------------+---------------+

→ Cada micro-lote trae eventos de TODO el período: el orden de 

### 4.5 Maestros: facturación, clientes, tickets e integridad referencial

In [11]:
b = raw["billing_monthly"]
(b.groupBy("currency")
  .agg(F.count("*").alias("facturas"), F.min("exchange_rate_to_usd").alias("fx_min"), F.max("exchange_rate_to_usd").alias("fx_max"),
       F.sum((F.col("subtotal") < 0).cast("int")).alias("subtotal_negativo"),
       F.sum(F.col("credits").isNull().cast("int")).alias("credits_nulos"))
  .orderBy("currency").show())
print("Meses facturados:", [r[0] for r in b.select("month").distinct().orderBy("month").collect()])

cu = raw["customers_orgs"]
cu.select(F.min("nps_score").alias("nps_min"), F.max("nps_score").alias("nps_max"),
          F.sum(((F.col("nps_score") < -100) | (F.col("nps_score") > 100)).cast("int")).alias("fuera_de_[-100,100]")).show()
cu.groupBy("lifecycle_stage").count().orderBy(F.desc("count")).show()

tk = raw["support_tickets"]
tk.groupBy("severity").agg(F.count("*").alias("tickets"),
                            F.round(F.avg(F.when(F.col("sla_breached") == "True", 1).otherwise(0)), 3).alias("tasa_sla_breach"),
                            F.round(F.avg("csat"), 2).alias("csat_prom")).orderBy("severity").show()
tk.select(F.min("csat"), F.max("csat")).show()

# Integridad referencial: ¿todos los org_id de las fuentes existen en el maestro de clientes?
orgs = cu.select("org_id")
for name, df in [("usage_events", ev)] + [(n, raw[n]) for n in ["users", "resources", "support_tickets", "billing_monthly", "nps_surveys", "marketing_touches"]]:
    huerfanos = df.select("org_id").distinct().join(orgs, "org_id", "left_anti").count()
    print(f"org_id sin maestro en {name:<18}: {huerfanos}")

+--------+--------+-------+-------+-----------------+-------------+
|currency|facturas| fx_min| fx_max|subtotal_negativo|credits_nulos|
+--------+--------+-------+-------+-----------------+-------------+
|     ARS|      51|0.00133|0.00162|                2|           34|
|     EUR|      29| 0.9981|1.19808|                1|           14|
|     USD|     160|0.85463|1.11791|               10|           89|
+--------+--------+-------+-------+-----------------+-------------+

Meses facturados: ['2025-06-01', '2025-07-01', '2025-08-01']
+-------+-------+-------------------+
|nps_min|nps_max|fuera_de_[-100,100]|
+-------+-------+-------------------+
|  -38.0|  101.0|                  1|
+-------+-------+-------------------+

+---------------+-----+
|lifecycle_stage|count|
+---------------+-----+
|         active|   54|
|        at_risk|   11|
|        churned|    6|
|       prospect|    6|
|           lead|    3|
+---------------+-----+

+--------+-------+---------------+---------+
|severity

## 5. Landing → Bronze (Parquet particionado)

Reglas de Bronze aplicadas:
- **Mismo grano** que la fuente (no se agrega nada).
- **Tipificación explícita** (esquemas de la sección 3) y columnas técnicas **`ingest_ts`** y **`source_file`**.
- **Deduplicación** por clave natural (`event_id`, `invoice_id`, etc.).
- **Particionamiento**: eventos por `event_date` (las consultas siempre filtran por rango de fechas); facturación por `billing_month`; maestros chicos por `ingest_date` (snapshot del día).
- **No** se particiona eventos por `service` además de fecha: con este volumen generaría 360 carpetas de pocos KB (*small files*). Se documenta como decisión reversible si el volumen crece.
- `coalesce` por partición para no generar un archivo por tarea.

In [12]:
INGEST_TS = F.current_timestamp()
INGEST_DATE = F.to_date(F.current_timestamp())

bronze_events = (ev_typed
    .dropDuplicates(["event_id"])
    .withColumn("ingest_ts", INGEST_TS)
    .select("event_id", "event_ts", "event_date", "org_id", "resource_id", "service", "region", "metric",
            "value_raw", "value_num", "unit", "cost_usd_increment", "schema_version", "carbon_kg", "genai_tokens",
            "ingest_ts", "source_file"))

NATURAL_KEYS = {"customers_orgs": "org_id", "users": "user_id", "resources": "resource_id", "support_tickets": "ticket_id",
                "marketing_touches": "touch_id", "billing_monthly": "invoice_id", "nps_surveys": None}

def to_bronze_master(name):
    df = raw[name]
    if NATURAL_KEYS[name]:
        df = df.dropDuplicates([NATURAL_KEYS[name]])
    df = df.withColumn("ingest_ts", INGEST_TS).withColumn("source_file", F.input_file_name())
    if name == "billing_monthly":
        return df.withColumn("billing_month", F.date_format(F.to_date("month"), "yyyy-MM")), "billing_month"
    return df.withColumn("ingest_date", INGEST_DATE), "ingest_date"

def write_bronze():
    (bronze_events.repartition("event_date")          # 1 archivo por fecha, sin miles de archivos chicos
        .write.mode("overwrite").partitionBy("event_date")
        .parquet(f"{CONFIG['bronze']}/usage_events"))
    for name in SCHEMAS:
        df, part = to_bronze_master(name)
        df.coalesce(1).write.mode("overwrite").partitionBy(part).parquet(f"{CONFIG['bronze']}/{name}")

write_bronze()
print("Bronze escrito en", CONFIG["bronze"])

Bronze escrito en /content/datalake/bronze


In [13]:
# Evidencia: rutas, cantidad de archivos y tamaños por dataset
def tree_stats(path):
    files = [f for f in glob.glob(path + "/**/*.parquet", recursive=True)]
    parts = {os.path.dirname(f) for f in files}
    return len(parts), len(files), round(sum(os.path.getsize(f) for f in files) / 1024, 1)

print(f"{'dataset bronze':<22}{'particiones':>12}{'archivos':>10}{'KB':>10}")
bronze_stats = {}
for d in sorted(os.listdir(CONFIG["bronze"])):
    p, f, kb = tree_stats(f"{CONFIG['bronze']}/{d}")
    bronze_stats[d] = {"particiones": p, "archivos": f, "kb": kb}
    print(f"{d:<22}{p:>12}{f:>10}{kb:>10}")
print("\nEjemplo de rutas:")
for x in sorted(glob.glob(f"{CONFIG['bronze']}/usage_events/*"))[:3]: print("  ", x.replace(BASE, ""))
print(f"\nCompresión: eventos JSON en Landing ≈ {ev_files[2]:,.0f} KB → Parquet en Bronze ≈ {bronze_stats['usage_events']['kb']:,.0f} KB")

dataset bronze         particiones  archivos        KB
billing_monthly                  3         3      19.6
customers_orgs                   1         1       6.7
marketing_touches                1         1      22.7
nps_surveys                      1         1       3.9
resources                        1         1       9.8
support_tickets                  1         1      18.7
usage_events                    60        60    2506.6
users                            1         1      23.4

Ejemplo de rutas:
   /datalake/bronze/usage_events/event_date=2025-07-03
   /datalake/bronze/usage_events/event_date=2025-07-04
   /datalake/bronze/usage_events/event_date=2025-07-05

Compresión: eventos JSON en Landing ≈ 12,624 KB → Parquet en Bronze ≈ 2,507 KB


In [14]:
# Idempotencia: re-ejecutar la carga NO duplica (overwrite dinámico por partición + dedupe por clave natural)
antes = spark.read.parquet(f"{CONFIG['bronze']}/usage_events").count()
write_bronze()
despues = spark.read.parquet(f"{CONFIG['bronze']}/usage_events").count()
print(f"usage_events en Bronze → antes: {antes:,} | después de re-ejecutar: {despues:,} | ¿idéntico? {antes == despues}")

spark.read.parquet(f"{CONFIG['bronze']}/usage_events").printSchema()

usage_events en Bronze → antes: 43,200 | después de re-ejecutar: 43,200 | ¿idéntico? True
root
 |-- event_id: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- org_id: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value_raw: string (nullable = true)
 |-- value_num: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- event_date: date (nullable = true)



## 6. Flujo batch de referencia: MapReduce ↔ Spark

Transformación de referencia del documento de diseño: **costo y requests diarios por organización y servicio** (base del mart `org_daily_usage_by_service`).

| Fase MapReduce | En este caso |
|---|---|
| InputSplit / RecordReader | Cada partición Parquet de Bronze (por `event_date`) |
| **Map** | evento → `((org_id, event_date, service), (costo, requests, 1))` |
| Combiner | suma parcial dentro de cada partición (reduce el shuffle) |
| Partitioner + **Shuffle/Sort** | hash de la clave → todas las filas de una misma clave van al mismo reducer |
| **Reduce** | suma de costos, requests y cantidad de eventos por clave |
| OutputFormat | Parquet (en Gold, segunda entrega) |

In [15]:
bronze = spark.read.parquet(f"{CONFIG['bronze']}/usage_events")

# --- Versión "MapReduce" con la API de RDD ---
def mapper(r):
    requests = r.value_num if (r.metric == "requests" and r.value_num is not None) else 0.0
    return ((r.org_id, str(r.event_date), r.service), (r.cost_usd_increment or 0.0, requests, 1))

def reducer(a, b):
    return (a[0] + b[0], a[1] + b[1], a[2] + b[2])

rdd_result = bronze.rdd.map(mapper).reduceByKey(reducer)   # reduceByKey aplica combiner del lado del map
print("Ejemplo de salida del reduce:", rdd_result.take(2))
print("Claves (org, fecha, servicio):", rdd_result.count())
print("\nLinaje del RDD (el '+-' marca el límite de stage = shuffle):")
print(rdd_result.toDebugString().decode())

Ejemplo de salida del reduce: [(('org_0lzjjege', '2025-08-29', 'networking'), (1.7736999999999998, 127.0, 3)), (('org_ykl1cq99', '2025-08-29', 'storage'), (2.6801, 141.0, 3))]
Claves (org, fecha, servicio): 11050

Linaje del RDD (el '+-' marca el límite de stage = shuffle):
(2) PythonRDD[652] at RDD at PythonRDD.scala:57 []
 |  MapPartitionsRDD[649] at mapPartitions at PythonRDD.scala:169 []
 |  ShuffledRDD[648] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[647] at reduceByKey at /tmp/ipykernel_2243/2661904605.py:11 []
    |  PythonRDD[646] at reduceByKey at /tmp/ipykernel_2243/2661904605.py:11 []
    |  MapPartitionsRDD[645] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  MapPartitionsRDD[644] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  SQLExecutionRDD[643] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  MapPartitionsRDD[642] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  MapPartitionsRDD[641] at javaTo

In [16]:
# --- Misma lógica con DataFrames (la API que se usará en el pipeline) ---
df_result = (bronze.groupBy("org_id", "event_date", "service")
             .agg(F.round(F.sum("cost_usd_increment"), 4).alias("daily_cost_usd"),
                  F.sum(F.when(F.col("metric") == "requests", F.col("value_num"))).alias("requests"),
                  F.count("*").alias("eventos")))

print("Plan físico: 'HashAggregate(partial)' = combiner · 'Exchange hashpartitioning' = shuffle · 'HashAggregate(final)' = reduce")
df_result.explain()
print("Claves en DataFrame:", df_result.count(), "| coincide con RDD:", df_result.count() == rdd_result.count())
df_result.orderBy(F.desc("daily_cost_usd")).show(5)

Plan físico: 'HashAggregate(partial)' = combiner · 'Exchange hashpartitioning' = shuffle · 'HashAggregate(final)' = reduce
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[org_id#3234, event_date#3248, service#3236], functions=[sum(cost_usd_increment#3242), sum(CASE WHEN (metric#3238 = requests) THEN value_num#3240 END), count(1)])
   +- Exchange hashpartitioning(org_id#3234, event_date#3248, service#3236, 8), ENSURE_REQUIREMENTS, [plan_id=5151]
      +- HashAggregate(keys=[org_id#3234, event_date#3248, service#3236], functions=[partial_sum(cost_usd_increment#3242), partial_sum(CASE WHEN (metric#3238 = requests) THEN value_num#3240 END), partial_count(1)])
         +- FileScan parquet [org_id#3234,service#3236,metric#3238,value_num#3240,cost_usd_increment#3242,event_date#3248] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/datalake/bronze/usage_events], PartitionFilters: [], PushedFilters: [], ReadSchema

## 7. Resumen y evidencia exportada

In [17]:
resumen = {
    "generado_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
    "spark_version": spark.version,
    "conteos_landing": conteos,
    "nulos": perfil["nulos"],
    "eventos": {
        "v2_desde": str(ev.filter("schema_version = 2").agg(F.min("event_date")).collect()[0][0]),
        "rango": [str(x) for x in ev.agg(F.min("event_date"), F.max("event_date")).collect()[0]],
        "value_sin_unit": sin_unit, "costos_negativos": negativos, "costos_bajo_tolerancia": bajo_tol,
        "percentiles_costo": dict(zip(["p1", "p25", "p50", "p75", "p99", "p99_9"], [round(x, 4) for x in q])),
        "event_id_duplicados": dups,
    },
    "bronze": bronze_stats,
}
with open(f"{CONFIG['evidence']}/01_profile_summary.json", "w") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False, default=str)
por_archivo.toPandas().to_csv(f"{CONFIG['evidence']}/01_event_time_range_by_file.csv", index=False)
print(json.dumps(resumen["eventos"], indent=2, ensure_ascii=False))
print("\nEvidencia guardada en", CONFIG["evidence"])

/tmp/ipykernel_2243/3966712390.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generado_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),


{
  "v2_desde": "2025-07-18",
  "rango": [
    "2025-07-03",
    "2025-08-31"
  ],
  "value_sin_unit": 2038,
  "costos_negativos": 216,
  "costos_bajo_tolerancia": 211,
  "percentiles_costo": {
    "p1": 0.0,
    "p25": 0.1285,
    "p50": 1.0043,
    "p75": 6.1153,
    "p99": 16.6934,
    "p99_9": 103.6077
  },
  "event_id_duplicados": 0
}

Evidencia guardada en /content/evidence


### Hallazgos que alimentan el diseño
1. **Evolución de esquema**: `schema_version = 2` empieza el 2025-07-18; `carbon_kg` aparece en todos los v2 y `genai_tokens` solo en `genai`. Silver debe unificar v1/v2 con columnas nulas para v1, no descartar v1.
2. **Tipos ambiguos**: parte de `value` llega como texto. Todos castean a número; se guarda `value_raw` para trazabilidad.
3. **Calidad**: eventos con `value` sin `unit`, costos negativos (la mayoría por debajo de −0.01) y spikes muy lejos del p99 → reglas de calidad + quarantine + flag de anomalía. Un umbral **global** no sirve (z-score marca casi nada, MAD marca demasiado): el costo depende del servicio y la métrica, así que el umbral de anomalía se calcula **por grupo**. El método y el umbral exactos se calibran en la segunda entrega.
4. **Late data**: cada archivo trae eventos de todo el período → el watermark no puede ser corto; la idempotencia se resuelve con dedupe por `event_id` + upserts por clave natural en Gold/Cassandra.
5. **Facturación multimoneda** (USD/ARS/EUR) con subtotales negativos y créditos nulos → normalización a USD en Silver.
6. **Integridad referencial completa**: todos los `org_id` existen en el maestro → los joins de enriquecimiento no pierden filas.
7. **Volumen chico** (≈13 MB): Spark se justifica por el diseño escalable (ver 5V en el documento), no por el tamaño de la muestra. Se controlan *small files* con `repartition`/`coalesce`.